# <center> <b> <span style="color:orange;"> AIMS RWANDA - Preparatory Class</span> </b></center>
# <center><b><span style="color:blue;">Lesson 2 : Higher-Order Functions, File I/O, and NumPy for Scientific Computing</span></b></center>

## Welcome back!
Last time (**Lesson 1**), you added two powerful tools to your kit:
- **dictionaries**: `key: value` lookups
- **functions**: `def`, parameters, default values, `return`, `*args`, `**kwargs`, `lambda`, recursion, and modules
  
Today's session has **four main parts**:

**1. Higher-order functions.** Functions that take *other functions* as input, or return them as output. We will look closely at `map()`, `filter()`, and `reduce()`, and then rebuild them ourselves.

**2. A first taste of file I/O.** Every program you have written so far lives and dies inside one notebook cell. Today you will read a real text file from disk and process its contents line by line.

**3. NumPy fundamentals.** Everything in Part 1 was pure Python: a `for` loop hiding inside `map`, `filter`, or a comprehension. That is completely fine for small tasks, but real scientific and data-science code needs to work on millions of numbers at once, think vectors, matrices, datasets. We introduce **NumPy**, the library that makes this fast, and connect it back to the vectorized thinking you already practiced in Part 1.

**4. Multi-dimensional arrays and Linear Algebra in practice.** We put NumPy to work on something that actually looks like data: a table of student grades across several courses. Along the way you will meet **axes**, the `.T` transpose, **vector norms**, and how to **solve a system of linear equations** with `np.linalg.solve`, the same systems you already solve by hand in your Linear Algebra course, now solved in one line.

By the end of today you will be able to:

1. Explain what makes a function "higher-order"
2. Use `map()`, `filter()`, and `functools.reduce()` confidently
3. Rebuild `map`, `filter`, and `reduce` yourself, using only a loop.
4. Open, read, and process a text file line by line
5. Explain why NumPy arrays are faster than Python lists for numeric work
6. Create, index, reshape, and do basic linear algebra with NumPy arrays
7. Recognize *vectorization* as the same idea behind `map()`, applied to numerical computing
8. Work with **2-D and 3-D arrays**, and reason about **axes** and **shape** for real, table-like data
9. Compute **vector norms**, use the **transpose**, and **solve a linear system** with NumPy
10. Combine everything above to solve a genuine text- and number-processing problem

## Warm-up: functions as values
Quick reminder from Lesson 1: in Python, a function is not special. It is a **value**, just like `5` or `"hello"`. That means you can:
- assign it to a variable
- store it in a list or dictionary- pass it as an **argument** to another function
- **return** it from another function
  
Let's see this concretely before we name it.

In [3]:
def shout(text):
    return text.upper() + "!"

# a function, stored in a variable, just like any other value
my_function = shout
print(my_function("hello"))
print(type(shout))

HELLO!
<class 'function'>


What do you predict `type(shout)` prints? Was it what you expected?

Now: what happens if we put `shout` itself, not `shout()`  inside a list?

In [4]:
toolbox = [shout, str.upper, len]
print(toolbox)


for tool in toolbox:
    print(tool("aims rwanda"))

[<function shout at 0x000001B9AE6DD1C0>, <method 'upper' of 'str' objects>, <built-in function len>]
AIMS RWANDA!
AIMS RWANDA
11


No parentheses on `shout`, `str.upper`, or `len` in the list, we are storing the *function itself*, not its result. The parentheses only appear when we actually **call** it, one line down.This idea, a function you can pass around like any other value, is the entire foundation of today's lesson.
> **Definition.** A **higher-order function** is a function that does at least one of:

> - takes another function as an **argument**, or
> - **returns** a function as its result.`map()`, `filter()`, and `reduce()` are the three classic examples, and you will use them constantly once they click.

## 1. `map()`: Apply a function to every element
Imagine you have a list of course names and, like in Lesson 1, you want the length of each one. Last time you wrote this with a `for` loop, then with a list comprehension. Here is a third way.

In [5]:
courses = ["Python", "Linear Algebra", "Statistics"]

lengths = map(len, courses)
print(lengths)          # what type is this??
print(list(lengths))    # force it to reveal its contents

[6, 14, 10]


Two things to notice:
1. `map(len, courses)` does **not** print a list. It prints something like `<map object at 0x...>`. `map` is *lazy*: it doesn't compute anything until you ask it to, by wrapping it in `list(...)` (or looping over it).
2. We passed `len` **without parentheses**. We are handing `map` the function itself, and `map` will call it on our behalf, once per element.

**Syntax:** `map(function, iterable)`: applies `function` to every item of `iterable`, one at a time, and returns a *map object* (convert to `list()` to see the results). 

In [6]:
def celsius_to_fahrenheit(c):
    return c * 9/5 + 32

temperatures_c = [0, 10, 20, 30, 37]
temperatures_f = list(map(celsius_to_fahrenheit, temperatures_c))
print(temperatures_f)

[32.0, 50.0, 68.0, 86.0, 98.6]


`map()` can also take a `lambda` directly, when the function is short enough that it isn't worth naming:

In [7]:
squares = list(map(lambda x: x**2, [1, 2, 3, 4, 5]))
print(squares)

[1, 4, 9, 16, 25]


**Your turn.** Using `map()` and a `lambda`, build a list `french_greetings` that adds `"Bonjour, "` in front of each name in `names`.

In [8]:
names = ["Uwase", "Eric", "Amina", "Grace"]

# french_greetings = ...
# print(french_greetings)

## 2. `filter()` : keep only what passes a test`map()` transforms every element. 
`filter()` instead **keeps or discards** each element, based on a function that returns `True` or `False`.

**Syntax:** `filter(function, iterable)`: keeps only the items of `iterable` for which `function(item)` is `True`.

In [9]:
def is_long(word):
    return len(word) > 6

words = ["cat", "elephant", "dog", "rhinoceros", "ant"]
long_words = list(filter(is_long, words))
print(long_words)

['elephant', 'rhinoceros']


Just like `map`, `filter` happily accepts a `lambda`:

In [10]:
numbers = list(range(1, 21))
evens = list(filter(lambda n: n % 2 == 0, numbers))
print(evens)

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


**Challenge:** write `filter_words(words, n)` that returns only the words strictly longer than `n`, using `filter()`.

In [11]:
def filter_words(words, n):
    return list(filter(lambda w: len(w) > n, words))

# Test:
print(filter_words(["Python", "AI", "Rwanda", "ML", "Statistics"], 3))

['Python', 'Rwanda', 'Statistics']


## 3. `reduce()`

Collapse a list into a single value`map` and `filter` are built into Python. `reduce` is *not* , it lives in the `functools` module, and needs an `import`.`reduce` takes a list and repeatedly combines its elements, two at a time, until only **one** value is left.

**Syntax:** `reduce(function, iterable)`, where `function` takes **two** arguments and returns **one** combined value.

In [12]:
from functools import reduce

numbers = [3, 7, 2, 9, 4]

total = reduce(lambda a, b: a + b, numbers)
print(total)

25


Let's trace exactly what happened, step by step:```reduce(lambda a, b: a + b, [3, 7, 2, 9, 4])```

- step 1:  a=3, b=7   -> 3 + 7   = 10
- step 2:  a=10, b=2  -> 10 + 2  = 12
- step 3:  a=12, b=9  -> 12 + 9  = 21
- step 4:  a=21, b=4  -> 21 + 4  = 25

````reduce```` carries a **running total** ("accumulator") through the list, combining it with each new element in turn.

What do you predict `reduce(lambda a, b: a if a > b else b, numbers)` computes? Try it.

In [13]:
numbers = [3, 7, 2, 9, 4]

biggest = reduce(lambda a, b: a if a > b else b, numbers)
print(biggest)

9


**Challenge:** write `max_in_list(numbers)` using `reduce()`, without using the built-in `max()`.

In [14]:
from functools import reduce

def max_in_list(numbers):
    return reduce(lambda a, b: a if a > b else b, numbers)

# Test:
print(max_in_list([5, 12, 3, 47, 8]))

47


`reduce` also accepts an **optional third argument**: a starting value for the accumulator. This matters a lot for empty lists!

What do you predict happens if you call `reduce(lambda a, b: a + b, [])` with **no** starting value? Try it and read the error message carefully.

In [15]:
from functools import reduce

# Uncomment and run to see the error:
#reduce(lambda a, b: a + b, [])

# Now with a starting value of 0, it works even on an empty list:
print(reduce(lambda a, b: a + b, [], 0))

0


**Summary so far**

| Function | Question it answers | Returns |
|---|---|---
|`map(f, L)` | "What does `f` do to *each* item?" | one new value **per item** |
|`filter(f, L)` | "Which items does `f` say yes to?" | a **subset** of the original items |
| `reduce(f, L)` | "How do all items combine into *one*?" | a **single** value |

All three are lazy or nearly so, and all three take the function **without** calling it, no parentheses.

## 4. Building `map`, `filter`, and `reduce` yourself

In [16]:
def my_map(function, iterable):
    result = []
    for item in iterable:
        result.append(function(item))
    return result

# Test against the real map():
print(my_map(lambda x: x * 2, [1, 2, 3]))
print(list(map(lambda x: x * 2, [1, 2, 3])))   # should match!

[2, 4, 6]
[2, 4, 6]


In [17]:
def my_filter(function, iterable):
    result = []
    for item in iterable:
        if function(item):
            result.append(item)
    return result

# Test:
print(my_filter(lambda x: x % 2 == 0, range(10)))

[0, 2, 4, 6, 8]


`reduce` is the trickiest of the three, because it needs to handle the optional starting value. 

Can you explain in your own words what the `if start is None` branch is doing, and why?

In [18]:
def my_reduce(function, iterable, start=None):
    iterator = iter(iterable)
    if start is None:
        accumulator = next(iterator)   # no starting value: use the first item
    else:
        accumulator = start
    for item in iterator:
        accumulator = function(accumulator, item)
    return accumulator

# Test:
print(my_reduce(lambda a, b: a + b, [1, 2, 3, 4]))       # 10
print(my_reduce(lambda a, b: a + b, [1, 2, 3, 4], 100))  # 110

10
110


## 5. A first taste of file I/O
Every program so far has lived entirely inside variables you typed by hand. 
Real data usually starts life in a **file**. Let's fix that.

First, let's create a small text file to work with, imagine this came from somewhere else (a download, a dataset, a scraped webpage):

In [19]:
lines_to_write = [
    "level\n",
    "civic\n",
    "python\n",
    "radar\n",
    "kayak\n",
    "statistics\n",
    "reviver\n",
    "rwanda\n",
]

with open("words.txt", "w") as f:
    f.writelines(lines_to_write)

print("File written!")

File written!


**The `with open(...) as f:` pattern.** 

This opens the file, gives you a handle `f` to work with, and, crucially **closes it automatically** when the indented block ends, even if an error occurs partway through. 

This is the standard, safe way to work with files in Python; always prefer it over manually calling `open()` and `close()`.`"w"` means *write mode* (creates the file, or overwrites it if it already exists). To read, we use `"r"` instead.

In [20]:
with open("words.txt", "r") as f:
    content = f.read()

print(content)
print(type(content))

level
civic
python
radar
kayak
statistics
reviver
rwanda

<class 'str'>


`f.read()` grabs the **entire file as one big string**, including the `\n` newline characters. Often you want it **line by line** instead:

In [21]:
with open("words.txt", "r") as f:
    for line in f:
        print(repr(line))   # repr() shows the \n so you can see it

'level\n'
'civic\n'
'python\n'
'radar\n'
'kayak\n'
'statistics\n'
'reviver\n'
'rwanda\n'


Notice every line still has a trailing `\n`. That is almost never what you want when you go on to process the word itself, so we typically call `.strip()` to remove it:

In [22]:
with open("words.txt", "r") as f:
    for line in f:
        word = line.strip()
        print(word, "->", len(word), "letters")

level -> 5 letters
civic -> 5 letters
python -> 6 letters
radar -> 5 letters
kayak -> 5 letters
statistics -> 10 letters
reviver -> 7 letters
rwanda -> 6 letters


**Challenge: a palindrome recognizer that reads from a file.** 

Read `words.txt` line by line and print only the palindromes.

In [23]:
def check_palindrome(word):
    return word == word[::-1]

with open("words.txt", "r") as f:
    for line in f:
        word = line.strip()
        if check_palindrome(word):
            print(word)

level
civic
radar
kayak
reviver


**Combining everything:** Build a sorted character-frequency table from the *entire file*.

In [24]:
def char_frequency_table(filename):
    with open(filename, "r") as f:
        text = f.read()
    text = text.replace("\n", "")   # ignore newline characters themselves
    freq = {char: text.count(char) for char in set(text)}
    # sort by frequency, highest first
    for char, count in sorted(freq.items(), key=lambda pair: pair[1], reverse=True):
        print(f"{char!r}: {count}")

char_frequency_table("words.txt")

'a': 7
'r': 5
'i': 5
'e': 4
't': 4
'v': 4
'c': 3
's': 3
'd': 2
'n': 2
'k': 2
'l': 2
'y': 2
'p': 1
'o': 1
'w': 1
'h': 1


---
## 7. From vectorized *thinking* to vectorized *computing*: introducing NumPy 

Look back at what you did in Part 1. 
When you wrote```pythonsquares = list(map(lambda x: x**2, [1, 2, 3, 4, 5]))```you were already thinking in a **vectorized** way: "apply this operation to the *whole collection* at once," rather than "loop and update one item at a time.

" `map()`, `filter()`, and comprehensions are Python's built-in way of expressing that idea, but under the hood, they are still a `for` loop in disguise, one Python function call per element.For small lists this is completely fine. But scientific computing, statistics, and machine learning often work with **thousands or millions** of numbers, and calling a Python function that many times, one at a time, is slow. This is exactly the problem **NumPy** (**Num**erical **Py**thon) was built to solve. NumPy gives you a new data type, the **array** (`ndarray`), and lets you apply an operation to an *entire* array in one line, executed internally in fast, compiled C code, not a Python loop at all.

Same idea you already know (map an operation over a collection), a much faster engine underneath.

In [25]:
import numpy as np


### 7.1 Creating your first arrays
The simplest way to build an array is from a Python list, using `np.array()`:

In [26]:
a = np.array([1, 2, 3, 4])
print(a)
print(type(a))

[1 2 3 4]
<class 'numpy.ndarray'>


NumPy also gives you ready-made functions to *generate* arrays, without typing every value by hand, very similar in spirit to `range()`, which you already know:

In [27]:
# arange(start, stop, step) - just like range(), but returns an array
c = np.arange(0, 10, 2)
print(c)

# linspace(start, stop, num) - num evenly spaced values between start and stop
d = np.linspace(0, 1, 5)
print(d)

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


### 7.2 Arrays vs. Python lists: why bother?
Let's compare `map()` on a list to the equivalent NumPy operation, on a **large** amount of data: one million numbers, each multiplied by 2.

In [28]:
numpy_array = np.arange(1000000)
python_list = list(range(1000000))

We will use the `%timeit` **magic command**: a Jupyter tool that reruns a line of code many times and reports its average execution time.

In [29]:
%timeit doubled = numpy_array * 2

7.99 ms ± 754 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [30]:
%timeit doubled = [num * 2 for num in python_list]

169 ms ± 36.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


You should see NumPy running **tens of times faster**, on exactly the same task. This is the same "apply one operation to everything" idea from `map()` and list comprehensions, but NumPy executes it as one instruction over the whole array in optimized C code, rather than one Python function call per element. This is what people mean by **vectorization**: replacing an explicit loop over elements with a single operation over the whole array.
> **Rule of thumb going forward:** if you find yourself writing `for element in array: ...` to do arithmetic, ask whether NumPy already has a vectorized way to say the same thing in one line.
### 7.3 Array math
Once your data lives in NumPy arrays, arithmetic works **elementwise**, automatically, no loop, no `map()`, needed:

In [31]:
x = np.array([[1, 2], [3, 4]], dtype=np.float64)
y = np.array([[5, 6], [7, 8]], dtype=np.float64)

print(x + y)      # elementwise sum
print(x * y)      # elementwise product (NOT matrix multiplication!)
print(np.sqrt(x)) # elementwise square root

[[ 6.  8.]
 [10. 12.]]
[[ 5. 12.]
 [21. 32.]]
[[1.         1.41421356]
 [1.73205081 2.        ]]


**Careful:** `x * y` above multiplies element by element, position by position. It is *not* the matrix product from linear algebra. For that, NumPy gives you `.dot()` (or the `@` operator):

In [32]:
v = np.array([9, 10])
w = np.array([11, 12])

# inner (dot) product of two vectors
print(v.dot(w))
print(np.dot(v, w))     # same thing, function form

# matrix - vector product
print(x.dot(v))

# matrix - matrix product
print(x.dot(y))

219
219
[29. 67.]
[[19. 22.]
 [43. 50.]]


This is your first real taste of **Linear Algebra in code**: vectors, matrices, and the dot product, exactly the objects you have been studying in your Linear Algebra course, now something you can compute directly instead of by hand.

### 7.4 Indexing, slicing, and reshaping
NumPy arrays support the same `[]` indexing and slicing you already know from lists, plus a few extra tricks. `reshape()` lets you rearrange the same data into a different shape, without changing the values themselves:

In [33]:
grid = np.arange(1, 10).reshape((3, 3))
print(grid)

[[1 2 3]
 [4 5 6]
 [7 8 9]]


In [34]:
values = np.arange(0, 10, 0.5)
mask = (values > 5) & (values < 7.5)   # note: & not "and", for elementwise comparison
print(mask)
print(values[mask])

[False False False False False False False False False False False  True
  True  True  True False False False False False]
[5.5 6.  6.5 7. ]


Compare this to how you would write "keep values strictly between 5 and 7.5" using `filter()` and a `lambda`, from Part 1. Which version do you find easier to read? Both are valid, NumPy's masking approach is simply the vectorized version of the same filtering idea, and it scales to huge arrays far better.

### 7.5 Vectorizing your own functions
What happens if you write an ordinary Python function, the kind you learned in Lesson 1, and hand it a whole array?

In [35]:
def step_function(x):
    """Scalar version: returns 1 if x >= 0, else 0."""
    if x >= 0:
        return 1
    else:
        return 0

values = np.array([-3, -2, -1, 0, 1, 2, 3])

# step_function(values)

The problem: `if x >= 0` doesn't make sense when `x` is an *entire array* rather than a single number, Python doesn't know how to treat an array as one `True`/`False` value.There are two ways to fix this. 

**Option 1**, let NumPy adapt your function automatically with `np.vectorize`:

In [36]:
step_function_vec = np.vectorize(step_function)
print(step_function_vec(values))

[0 0 0 1 1 1 1]


**Option 2** (usually faster): rewrite the function to be array-aware from the start, using a comparison that itself produces an array of booleans, then converting `True`/`False` into `1`/`0`:

In [37]:
def step_function(x):
    """Vector-aware version: works on a single number OR a whole array."""
    return 1 * (x >= 0)

print(step_function(values))
print(step_function(-1.2), step_function(2.6))   # still works on single numbers too!

[0 0 0 1 1 1 1]
0 1


### 7.6 Broadcasting: combining arrays of different shapes
One last idea, useful and a little surprising the first time you see it. Suppose you want to add the same vector to *every row* of a matrix. The loop-based way, using what you already know:

In [38]:
matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])
vector = np.array([1, 0, 1])

result = np.empty_like(matrix)
for i in range(4):
    result[i, :] = matrix[i, :] + vector

print(result)

[[ 2  2  4]
 [ 5  5  7]
 [ 8  8 10]
 [11 11 13]]


NumPy lets you skip the loop entirely, thanks to **broadcasting**: when shapes don't match exactly but are *compatible*, NumPy automatically stretches the smaller array to match the larger one, without actually copying any data in memory.

In [39]:
result = matrix + vector   # no loop at all!
print(result)

[[ 2  2  4]
 [ 5  5  7]
 [ 8  8 10]
 [11 11 13]]


Same result, one line, no loop. `matrix` has shape `(4, 3)` and `vector` has shape `(3,)`; NumPy treats `vector` as if it had been copied into every row, without ever actually doing so, which is both faster and more memory-efficient.

**Broadcasting rule, informally:** two shapes are compatible if, comparing them from the right, each pair of dimensions is either equal, or one of them is `1`.
### 7.7 Summary: your two toolkits

||Pure Python | NumPy (Part 3 today) |
|---|---|---|
| Collection type | `list` | `np.ndarray` |
| Apply to every element | `map()`, comprehension | arithmetic on the whole array, e.g. `arr * 2` |
| Keep elements matching a condition | `filter()` | boolean mask, e.g. `arr[arr > 5]` |
| Combine all elements into one | `reduce()` | `.sum()`, `.prod()`, `.dot()`, etc. |
| Speed on large data | one Python call per element | compiled C loop, often 10-100x faster |

Neither toolkit replaces the other; you now understand *why* NumPy exists, and you'll reach for it whenever your data is genuinely numeric and possibly large: matrices, vectors, datasets, anything you would compute by hand in your Linear Algebra or Statistics courses.

**Challenge.** Rewrite `celsius_to_fahrenheit()` from Part 1 (Section 1) so it works directly on a NumPy array, without `map()` and without `np.vectorize`, using only vectorized arithmetic.

In [40]:
def celsius_to_fahrenheit_np(c_array):
    ...

# Test:
# temperatures_c = np.array([0, 10, 20, 30, 37])
# print(celsius_to_fahrenheit_np(temperatures_c))

---
## 8 - Multi-dimensional arrays and Linear Algebra in practice
So far every array you built was **1-D**: a single row of numbers. Real datasets are almost never that simple. A spreadsheet of exam results, a grayscale image, a set of sensor readings over time, all of these are naturally **2-D or higher**. NumPy handles this without any new syntax: the same array type, just with more **axes**.We will use one running example for the rest of this part: a small table of grades for **5 students** across **4 courses** at AIMS. 

In [41]:
students = ["Uwase", "Eric", "Amina", "Grace", "Patrick"]
courses  = ["Python", "Linear Algebra", "Statistics", "Calculus"]

grades = np.array([
    [78, 85, 92, 66],   # Uwase
    [88, 74, 69, 91],   # Eric
    [95, 90, 88, 84],   # Amina
    [60, 72, 65, 70],   # Grace
    [82, 79, 95, 88],   # Patrick
])

print(grades)
print("shape:", grades.shape)

[[78 85 92 66]
 [88 74 69 91]
 [95 90 88 84]
 [60 72 65 70]
 [82 79 95 88]]
shape: (5, 4)


### 8.1 Shape and axes`grades.shape` is `(5, 4)`: **5 rows** (one per student) and **4 columns** (one per course). 

This pairing, `(rows, columns)`, is called the array's **shape**, and each dimension is called an **axis**: `axis=0` runs down the rows (across students), `axis=1` runs across the columns (across courses).What do you predict `grades[1]` gives you? What about `grades[:, 2]`?.

In [42]:
print(grades[1])      # Eric's row: all 4 of his grades
print(grades[:, 2])   # Statistics column: everyone's grade in that one course

[88 74 69 91]
[92 69 88 65 95]


`grades[1]` slices along axis 0 (**one student, all courses**). `grades[:, 2]` slices along axis 1 (**one course, all students**). The `:` means "give me everything along this axis."This "which axis?" question comes up constantly with aggregation functions like `.sum()`, `.mean()`, `.max()`. By default they collapse the *entire* array into one number:

In [43]:
print(grades.mean())     # the single average of all 20 grades

80.55


But you can tell NumPy which axis to collapse, using the `axis=` argument.

In [44]:
course_averages = grades.mean(axis=0)
for course, avg in zip(courses, course_averages):
    print(f"{course:16s}: {avg:.1f}")

Python          : 80.6
Linear Algebra  : 80.0
Statistics      : 81.8
Calculus        : 79.8


**Your turn.** Compute each *student's* average across all their courses, using `axis=1`, and print it next to their name.

In [45]:
# student_averages = grades.mean(axis=...)
# for student, avg in zip(students, student_averages):
#     print(f"{student:10s}: {avg:.1f}")

### 8.2 The transpose
Sometimes you want to flip rows and columns entirely, for example, to loop over courses first instead of students. NumPy's `.T` attribute does this instantly, with **no copying and no loop**:

In [46]:
print(grades.shape, "->", grades.T.shape)
print(grades.T)

(5, 4) -> (4, 5)
[[78 88 95 60 82]
 [85 74 90 72 79]
 [92 69 88 65 95]
 [66 91 84 70 88]]


`grades.T[2]` is now the *third course's* row, exactly the same values as `grades[:, 2]` from earlier, just accessed the other way round. The transpose is the array equivalent of turning a spreadsheet 90 degrees.Transposes matter a lot in linear algebra: many formulas, dot products between rows and columns, covariance matrices, solving equations, only make sense once shapes line up correctly, and `.T` is how you make that happen.

### 8.3 Vector norms: measuring the "size" of a student's grade profile

In Linear Algebra you have already met the idea of a vector's **norm**, roughly, "how big is this vector?" The most common one, the **Euclidean norm** (also called the $L_2$ norm), is$$\lVert v \rVert = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2}$$NumPy computes this directly with `np.linalg.norm`. Let's use it to compare two students' grade profiles, treating each student's row as a 4-dimensional vector:

In [47]:
uwase_grades = grades[0]
eric_grades = grades[1]

print("‖Uwase‖ =", np.linalg.norm(uwase_grades))
print("‖Eric‖  =", np.linalg.norm(eric_grades))

‖Uwase‖ = 161.64467204334326
‖Eric‖  = 162.05554603283406


On its own, the norm of one student's row is not very meaningful, it mixes "how high are the grades" with "how many courses." Norms become genuinely useful when you compare the **difference** between two vectors, which gives you a notion of *distance*: how far apart are two students' performance profiles?$$\lVert u - v \rVert = \text{distance between } u \text{ and } v$$

In [48]:
difference = uwase_grades - eric_grades
distance = np.linalg.norm(difference)
print("difference vector:", difference)
print("distance between Uwase and Eric:", distance)

difference vector: [-10  11  23 -25]
distance between Uwase and Eric: 37.080992435478315


**Your turn.** Using a loop (or, if you want a challenge, `np.linalg.norm` combined with broadcasting and no loop at all), find which **two students** have the *most similar* grade profiles, i.e. the smallest distance between their rows.

In [49]:
# Hint (loop version): compare every pair of students with nested loops,
# keep track of the smallest distance seen so far.

best_pair = None
best_distance = None

for i in range(len(students)):
    for j in range(i + 1, len(students)):
        d = np.linalg.norm(grades[i] - grades[j])
        if best_distance is None or d < best_distance:
            best_distance = d
            best_pair = (students[i], students[j])

print(best_pair, "distance:", best_distance)

('Amina', 'Patrick') distance: 18.841443681416774


### 8.4 Solving a system of linear equations
Here is a genuinely practical problem, the kind of thing `np.linalg.solve` was built for.

Suppose the four courses are **not weighted equally** toward a student's final AIMS score. You know the *final scores* for three students, and you know their per-course grades, but you don't know the **weights** the program uses for each course. Can you recover them?

This is exactly a system of linear equations: if $w_1, w_2, w_3, w_4$ are the unknown weights for the four courses, then for each student,$$w_1 \cdot \text{Python} + w_2 \cdot \text{LinAlg} + w_3 \cdot \text{Stats} + w_4 \cdot \text{Calc} = \text{final score}$$With **4 unknowns**, we need **4 equations**, so let's use four students' grades and their (given) final scores:

In [58]:
# Coefficient matrix: each row is one student's grades across the 4 courses
A = grades[:4]   # first 4 students, shape (4, 4) -- a square system

# Known final scores for those same 4 students
final_scores = np.array([81.2, 79.6, 89.6, 66.8])

print("A =")
print(A)
print("final_scores =", final_scores)

A =
[[78 85 92 66]
 [88 74 69 91]
 [95 90 88 84]
 [60 72 65 70]]
final_scores = [81.2 79.6 89.6 66.8]


In [59]:
weights = np.linalg.solve(A, final_scores)
for course, w in zip(courses, weights):
    print(f"{course:16s}: weight = {w:.3f}")

Python          : weight = 0.257
Linear Algebra  : weight = 0.310
Statistics      : weight = 0.241
Calculus        : weight = 0.191


`np.linalg.solve(A, b)` finds the vector $w$ such that $A w = b$.

Let's sanity-check the result: multiplying `A` by our recovered `weights` should reproduce the original final scores.

In [52]:
check = A.dot(weights)
print("recomputed scores:", check)
print("original scores:  ", final_scores)
print("all close?", np.allclose(check, final_scores))

recomputed scores: [81.2 79.6 89.6 66.8]
original scores:   [81.2 79.6 89.6 66.8]
all close? True


`np.allclose` is the right way to compare floating-point results, computers store decimals with tiny rounding errors, so `==` can fail even when two numbers are "the same" for all practical purposes.

### Home exercise -  **(higher-order functions + file I/O).** 

Write `find_hapaxes(filename)`: a function that reads a text file and returns a list of every **hapax**, a word that occurs *exactly once* in the whole file.

Suggested steps:
1. Read the file and split it into words (`.split()` on whitespace works for a first version).
2. Build a frequency dictionary.
3. Use a comprehension or `filter()` to keep only the words whose count is exactly `1`.


Bonus: can you rewrite step 2 as a `reduce()` instead of a comprehension?

In [55]:
def find_hapaxes(filename):
    ...

# Test:
# print(find_hapaxes("words.txt"))

### Home exercise - **(NumPy fundamentals)**

Write `normalize(arr)`: a function that takes a 1-D NumPy array and returns a new array rescaled to the range `[0, 1]`, using the formula$$\text{normalized}_i = \frac{x_i - \min(x)}{\max(x) - \min(x)}$$Do this **without any loop**, using only vectorized NumPy operations (`arr.min()`, `arr.max()`, and ordinary arithmetic).

Bonus: what does your function do if every value in `arr` is the same? Can you make it handle that case without crashing?

In [56]:
def normalize(arr):
    ...

# Test:
# print(normalize(np.array([2, 4, 4, 4, 5, 5, 7, 9])))

### Home exercise: **(multi-dimensional arrays & Linear Algebra)**

Using the `grades`. Write `standardize(grades)`, a function that returns a new array where each **column** (course) has been rescaled to have mean `0` and standard deviation `1`. This is called **z-score standardization**, a very common preprocessing step in statistics and machine learning:$$z_{ij} = \frac{x_{ij} - \text{mean of column } j}{\text{std of column } j}$$   Hint: `grades.mean(axis=0)` and `grades.std(axis=0)` give you one value per course; broadcasting does the rest.2. Write `most_consistent_student(grades)`: using `np.std` with the right `axis`, find and return the name of the student whose grades have the **smallest spread** (standard deviation) across their 4 courses, the most "consistent" performer, regardless of their average.3. 


In [57]:
def standardize(grades):
    ...

def most_consistent_student(grades):
    ...

# Test:
# print(standardize(grades))
# print(most_consistent_student(grades))
# new_weights = np.array([0.3, 0.3, 0.2, 0.2])
# print(grades.dot(new_weights))